# Linear Regression via Gradient Descent

## Initial Setup

Suppose we have training data $\{(x_{11},\ldots, x_{1p}, y_1), \ldots, (x_{n1},\ldots, x_{np}, y_n)\}$.  So $(x_{i1}, \ldots x_{ip})$ is a feature observation and $y_i$ is the associated label.  We will assume that $x_{ij}$ and $y_i$ are all real numbers (i.e. continuous).  The linear regression model is defined as:

$$
\hat y_i = \beta_0 + \sum_{k=1}^{p} x_{ik}\,\beta_k
$$

It is both mathematically useful and useful for implementing efficient vectorized code with packages like **numpy** and **PyTorch** to express the above set of $n$ equations as a matrix computation. Let $\beta = (\beta_0, \beta_1, \dots, \beta_p)^{T}$. Futhermore, arrange the training data into a matrice $X$ and $Y$ as follows (note the column of $1$s at the beginning of $X$ to absorb the intercept $\beta_{0}$)

$$
X = \begin{bmatrix} 1 & x_{11} & \cdots & x_{1p} \\ \vdots & \vdots & & \vdots \\ 1 & x_{n1} & \cdots & x_{np} \end{bmatrix}
$$

$$
Y = \begin{bmatrix} \, y_1 \, \\ \vdots \\ \, y_n \, \end{bmatrix}
$$


Now we can compute the a $n \times 1$ matrix of all the predictions $\hat{y_i}$ as with the following matrix computation:

$$
X \beta = \begin{bmatrix} \, \hat y_1 \, \\ \vdots \\ \, \hat y_n  \, \end{bmatrix}
$$

To borrow some language from deep learning, we will sometimes refer to $X \beta$ as a *full-batch forward pass* of the model.  We will sometimes use the following notation $\hat{Y} = X \beta$.

## Generating Synthetic Data

Let's generate some synthetic data to illustrate our derivation.

In [1]:
import numpy as np

In [2]:
rng = np.random.default_rng(0)
n = 500
beta = np.array([1, -1, -0.5, 1.5])
intercept = 0
sigma = 1.0

X_inputs = rng.normal(size=(n, beta.shape[0]))
y = intercept + X_inputs @ beta + rng.normal(scale=sigma, size=n)

## Implementing Data Matrix and Parameters in Code

Below we have that $\beta_0=$ `bias` and $(\beta_1, \dots, \beta_p)=$ `weights`.  Again we are using terms that foreshadow neural netorks.

In [3]:
bias = rng.normal(scale=1, size=1)
weights = rng.normal(scale=1, size=4)
print(weights)
print(bias)

[-2.19653221 -0.89302575 -0.51095075  1.47924899]
[-0.85843593]


It will be useful concatenat the `weight` and `bias` into a single variable `parameters`$=(\beta_0, \beta_1, \dots, \beta_p) = \beta^T$.

In [4]:
parameters = np.append(bias, weights)
parameters

array([-0.85843593, -2.19653221, -0.89302575, -0.51095075,  1.47924899])

Let's now construct our data matrix `X`$=X$.

In [5]:
ones = np.ones((X_inputs.shape[0], 1), dtype=X_inputs.dtype)
X = np.hstack((ones, X_inputs))
X

array([[ 1.        ,  0.12573022, -0.13210486,  0.64042265,  0.10490012],
       [ 1.        , -0.53566937,  0.36159505,  1.30400005,  0.94708096],
       [ 1.        , -0.70373524, -1.26542147, -0.62327446,  0.04132598],
       ...,
       [ 1.        ,  1.85744412, -0.82073139, -0.14101172,  1.35408138],
       [ 1.        ,  1.73963397,  0.06523589,  1.24972527,  0.75063397],
       [ 1.        , -0.55581573, -2.01881162, -0.90942756,  0.36922933]],
      shape=(500, 5))

## Defining Residuals and Loss

We'll want a measure of how well the linear regression model is working for a given set of parameter values. Towards this end we define some additional notation:

1. The *residual* of the $i$th prediction will be $r_i$.
1. The overall *loss* for a give set of paramenters will be $L(\beta_0, \beta_1, \dots, \beta_p)$ - this is mean squared error.

The definitions are as follows

$$
r_i = \hat{y_i} - y_i
$$

$$
L(\beta_0, \beta_1, \dots, \beta_p)= \frac{1}{n}\sum_{i=1}^{n} r_i^2
$$

## Implementing the Loss in Code

The above definitions of residuals and loss are implemented in the following `loss()` function:

In [6]:
def loss(parameters, X, y):
    y_hat = X @ parameters
    residuals = (y_hat - y)
    return (residuals.T @ residuals) / len(y)

Regarding the `loss()` function:

1. `y_hat = X @ parameters` is the vector of predictions (forward pass) with the current state of the `parameters`: $X\beta$.
2. `residuals = y_hat - y` is the vector of residuals for all the predictions: $(r_1, \ldots, r_n)$.
3. `((residals).T @ (residuals)) / len(y)` is the mean squared error: $\frac{1}{n}\sum_{i=1}^{n} r_i^2$.

Notice that for #3 above what looks like matrix multiplication **numpy** syntax (`@`) is really just a dot product of the `residuals` vector with itself, which is a convenient/efficient way of calculating the mean-squared error.

In [7]:
loss(parameters, X, y)

np.float64(12.71763648826799)

## Deriving The Gradient of the Loss

The partial derivatives of the loss $L$ with respect to the parameters $\beta_j$ are as follows:

$$
\begin{aligned}
\frac{\partial\, L}{\partial \beta_0} &= \frac{1}{n}\sum_{i=1}^{n} 2\,r_i\,\frac{\partial r_i}{\partial \beta_0} = \frac{2}{n}\sum_{i=1}^{n} r_i \\[4pt]
\frac{\partial\, L}{\partial \beta_j} &= \frac{1}{n}\sum_{i=1}^{n} 2\,r_i\,\frac{\partial r_i}{\partial \beta_j} = \frac{2}{n}\sum_{i=1}^{n} x_{ij}\,r_i
\end{aligned}
$$

Let's now express this in matrix notatation.  First let's recall what the data matrix $X$ looks like:

$$
X = \begin{bmatrix} 1 & x_{11} & \cdots & x_{1p} \\ \vdots & \vdots & & \vdots \\ 1 & x_{n1} & \cdots & x_{np} \end{bmatrix}
$$

For the purposes of this let's let $X_{0}$ be the first column of all $1$s of $X$.  And let $X_{j}$ be the column corresponding to the $j$th feature.  Notice that for $j = 0, \ldots, p$ we have that:

$$
\sum_{i=1}^{n} x_{ij}\,r_i = X_j \cdot (r_1, \ldots r_n)^{T}
$$

which said in words is simply the dot product of the $j$th column of $X$ with the column-vector residuals.  

Let's take a closer look at $X^T$

$$
X^T = \begin{bmatrix} 1 & \cdots & 1 \\ x_{11} & \cdots & x_{1n} \\ \vdots & & \vdots \\ x_{p1} & \cdots & x_{pn} \end{bmatrix}
$$

note the $X$ is tall and skinny $(n \times p)$ while $X^T$ is short and fat $(p \times n)$.  We can now express gradient vector as in term of the transpose of the data matrix, $X^{T}$, and the vector of residuals $(r_1, \ldots r_n)$ as follows:

$$
\nabla {L}(\beta_0, \ldots, \beta_p) = \frac{2}{n} X^{T} (r_1, \ldots r_n)^{T} = \frac{2}{n} \begin{bmatrix} \, \sum_{i=1}^{n} r_i  \, \\ \sum_{i=1}^{n} x_{i1}\,r_i \\ \vdots \\ \, \sum_{i=1}^{n} x_{ip}\,r_i \, \end{bmatrix}
$$

## Implementing the Gradient in Code

We implement the above derivation of the gradient in terms of matrix computations in the following `gradient()` function:

In [8]:
def gradient(parameters, X, y):
    residuals = (X @ parameters - y)
    return (2 / len(y)) * (X.T @ residuals) 

In [9]:
gradient(parameters, X, y)

array([-1.6254411 , -6.77614866,  0.70044759, -0.28145517, -0.33059282])

## Deriving the Parameter Update

If $\eta$ is our learning rate, then we have that our parameter update step looks like:

$$
\begin{aligned}
\frac{\partial L}{\partial \beta_0} &= \frac{2}{n}\sum_{i=1}^{n} r_i, &
\beta_0 &\leftarrow \beta_0 - \eta\,\frac{2}{n}\sum_{i=1}^{n} r_i \\[4pt]
\frac{\partial L}{\partial \beta_j} &= \frac{2}{n}\sum_{i=1}^{n} x_{ij}\,r_i, &
\beta_j &\leftarrow \beta_j - \eta\,\frac{2}{n}\sum_{i=1}^{n} x_{ij}\,r_i
\end{aligned}
$$

Written in matrix notation we have that:

$$
(\beta_0, \ldots \beta_p) \leftarrow (\beta_0, \ldots \beta_p) - \eta * \nabla {L}(\beta_0, \ldots, \beta_p)
$$

## Implementing the Parameter Update - Fitting the Model

The parameter update procedure is implemented in the following `for` loop for 100 iterations (`epochs`).

In [10]:
learning_rate = 0.1
for epoch in range(100):
    grad = gradient(parameters, X, y)
    parameters += -learning_rate * grad

In [11]:
parameters

array([-0.0381367 ,  1.05298517, -1.0774468 , -0.45626473,  1.53068716])

In [12]:
[0, 1, -1, -0.5, 1.5]

[0, 1, -1, -0.5, 1.5]

## Comparing Gradient Descent to `sklearn`

Our gradient descent approach matches `sklearns` results almost exactly - kind of shockingly close.

In [13]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_inputs, y)
print(model.intercept_, model.coef_)

-0.03813669481819029 [ 1.05298517 -1.07744679 -0.45626473  1.53068716]


## Implementing In **PyTorch**

Let's implement the same procedure above using **Pytorch**.

In [14]:
import torch

First, we construct our feature and label observation matrices $X$ and $Y$.

In [15]:
X_inputs = torch.tensor(X_inputs, dtype=torch.float32)
ones = torch.ones((X_inputs.size(0), 1), dtype=X_inputs.dtype)
X = torch.cat([ones, X_inputs], dim=1)
y = torch.tensor(y, dtype=torch.float32)

Next, we initialize our parameters $\beta_0 , \ldots, \beta_p$ with small random values.

In [16]:
torch.manual_seed(0)
bias = torch.randn(size=(1,), dtype=torch.float32)
weights = torch.randn(size=(4,), dtype=torch.float32)

Concantinating together `bias` and `weights` to give us a variable `parameters=`$(\beta_0 , \ldots, \beta_p)$.

In [17]:
parameters = torch.cat((bias, weights), dim=0)
parameters

tensor([ 1.5410, -0.2934, -2.1788,  0.5684, -1.0845])

Now we can implement our `loss()` function $L(\beta_0 , \ldots, \beta_p) =  \frac{1}{n}\sum_{i=1}^{n} r_i^2$.

In [18]:
def loss(parameters, X, y):
    y_hat = X @ parameters
    residuals = (y_hat - y)
    return torch.dot(residuals, residuals) / len(y)

In [19]:
loss(parameters, X, y)

tensor(14.4666)

Next, we implement the `gradient()` (of `loss()`) function:

$$
\nabla {L}(\beta_0, \ldots, \beta_p) = \frac{2}{n} X^{T} (r_1, \ldots r_n)^{T} = \frac{2}{n} \begin{bmatrix} \, \sum_{i=1}^{n} r_i  \, \\ \sum_{i=1}^{n} x_{i1}\,r_i \\ \vdots \\ \, \sum_{i=1}^{n} x_{ip}\,r_i \, \end{bmatrix}
$$

In [20]:
def gradient(parameters, X, y):
    residuals = (X @ parameters - y)
    return (2 / len(y)) * (X.T @ residuals) 

In [21]:
gradient(parameters, X, y)

tensor([ 3.0662, -2.7991, -2.1738,  1.7792, -5.4237])

And finally we implement our parameter update logic for fitting iteratively:

$$
(\beta_0, \ldots \beta_p) \leftarrow (\beta_0, \ldots \beta_p) - \eta * \nabla {L}(\beta_0, \ldots, \beta_p)
$$

In [22]:
learning_rate = 0.1
for epoch in range(100):
    grad = gradient(parameters, X, y)
    parameters += -learning_rate * grad

In [23]:
parameters

tensor([-0.0381,  1.0530, -1.0774, -0.4563,  1.5307])

Let's compare our results to **skelearn**.

In [24]:
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(X_inputs, y)
print(model.intercept_, model.coef_)

-0.03813672 [ 1.0529853 -1.0774469 -0.4562649  1.530687 ]
